# HiC-ECC | Module 2: Compare
Hi-C comparison with CHESS, then unique region extraction.

## Config
**Edit only this cell.**

In [ ]:
import yaml

with open('../../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

TISSUES       = cfg['samples']
GENOME        = cfg['genome']
RESOLUTION    = cfg['resolution']
CHROMOSOMES   = cfg['chromosomes']
WINDOW        = cfg['comparison']['window']
CHESS_THREADS = cfg['comparison']['threads']
SN_THR        = cfg['comparison']['sn_threshold']
ZSIM_THR      = cfg['comparison']['zsim_threshold']
COOL_DIR      = f"{cfg['output_dir']}/cool_files/{RESOLUTION}"
COOL_SUFFIX   = f"_deephic.{RESOLUTION//1000}kb.cool"
OUT_DIR       = f"{cfg['output_dir']}/chess_output"
CHR_QUERY     = "chr4"    # chromosome for unique region extraction

## Setup

In [ ]:
import os, subprocess, itertools
import pandas as pd

BED_DIR    = os.path.join(OUT_DIR, "bed_files")
UNIQ_DIR   = os.path.join(OUT_DIR, "unique_regions")

for d in [OUT_DIR, BED_DIR, UNIQ_DIR]:
    os.makedirs(d, exist_ok=True)

WIN_LABEL = f"{WINDOW // 1000}kb"
print("Directories ready.")

# Section 1 (CHESS Comparison)

### 1a. Generate BED region pairs

In [ ]:
for chrom in CHROMOSOMES:
    bed = os.path.join(BED_DIR, f"{GENOME}_{chrom}_{WINDOW}_win_{RESOLUTION}_step.bed")
    if os.path.isfile(bed):
        print(f"[skip] BED exists: {chrom}")
        continue
    print(f"[pairs] {chrom}")
    subprocess.run(
        ["chess", "pairs", GENOME, str(WINDOW), str(RESOLUTION), bed, "--chromosome", chrom],
        check=True
    )

print("BED files ready.")

### 1b. Run pairwise chess sim (all unique pairs)

In [ ]:
pairs = list(itertools.combinations(TISSUES, 2))
print(f"{len(pairs)} tissue pairs × {len(CHROMOSOMES)} chromosomes = {len(pairs)*len(CHROMOSOMES)} comparisons")

for ref, comp in pairs:
    ref_cool  = os.path.join(COOL_DIR, ref,  ref  + COOL_SUFFIX)
    comp_cool = os.path.join(COOL_DIR, comp, comp + COOL_SUFFIX)

    for cool, label in [(ref_cool, ref), (comp_cool, comp)]:
        if not os.path.isfile(cool):
            print(f"[SKIP] Missing: {cool}")
            continue

    pair_dir = os.path.join(OUT_DIR, ref)
    os.makedirs(pair_dir, exist_ok=True)

    for chrom in CHROMOSOMES:
        bed = os.path.join(BED_DIR, f"{GENOME}_{chrom}_{WINDOW}_win_{RESOLUTION}_step.bed")
        out = os.path.join(pair_dir, f"{chrom}_{ref}_vs_{comp}{COOL_SUFFIX.replace('.cool','')}.tsv")
        print(f"[sim] {ref} vs {comp} | {chrom}")
        subprocess.run(
            ["chess", "sim", ref_cool, comp_cool, bed, out, "-p", str(CHESS_THREADS)],
            check=True
        )

print("All comparisons done.")

# Section 2 (Unique Region Extraction)
Set `CHR_QUERY` in the config cell, then run this section.

In [ ]:
bed_file = os.path.join(BED_DIR, f"{GENOME}_{CHR_QUERY}_{WINDOW}_win_{RESOLUTION}_step.bed")
regions  = pd.read_csv(bed_file, sep='\t', header=None)
print(f"Loaded {len(regions)} regions from {bed_file}")

for tissue in TISSUES:
    samp_sims = []

    # collect all TSV files involving this tissue
    for tissue_dir in TISSUES:
        tsv_dir = os.path.join(OUT_DIR, tissue_dir)
        if not os.path.isdir(tsv_dir):
            continue
        for f in os.listdir(tsv_dir):
            if f.endswith('.tsv') and CHR_QUERY in f:
                if f'_{tissue}_vs_' in f or f'_vs_{tissue}_' in f:
                    path = os.path.join(tsv_dir, f)
                    sim  = pd.read_csv(path, sep='\t', index_col=0)
                    filt = sim[(sim['SN'] >= SN_THR) & (sim['z_ssim'] <= ZSIM_THR)]
                    samp_sims.append(filt)

    if not samp_sims:
        print(f"[{tissue}] no data found")
        continue

    # intersect indices across all comparisons
    uniq_idx = set(samp_sims[0].index)
    for s in samp_sims[1:]:
        uniq_idx.intersection_update(s.index)

    out_bed = os.path.join(UNIQ_DIR, f"{GENOME}_{tissue}_{CHR_QUERY}_{WIN_LABEL}_{ZSIM_THR}.bed")
    tmp = regions.iloc[list(uniq_idx), :3].merge(
        samp_sims[0]['ssim'], how='inner', left_index=True, right_index=True
    ).rename(columns={0:'chr', 1:'start', 2:'end', 'ssim':'ssim_score'})
    tmp.to_csv(out_bed, sep='\t', header=False, index=False)
    print(f"[{tissue}] {len(tmp)} unique regions → {out_bed}")

print("Done.")